# DuckDB SQL — local usage examples

This notebook shows how to query ABHunter Apache Parquet databases **directly with DuckDB SQL**, without the AntibodySearchEngine.

Use this when you need custom aggregations (e.g. IGHV×IG(K/L)V co-occurrence) or unconstrained scans beyond the web/ASE search mask.
For the same filters as the web form, see `antibody_search_engine_examples.ipynb` in this folder.

**Data:** these notebooks live in `publication/DemoNotebooks/`. Toggle `USE_FULL_DB` in setup:
- `False` (default): `examples/minimal/` under the package root
- `True`: `data/{Heavy,Light,Paired}` under the package root

**Important:** sequence globs must **exclude** `metadata.parquet` (different schema: donor/file metadata). Join metadata only when you need `subject` / `total_sequences`.

## 0. Setup

In [14]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

# publication/DemoNotebooks/ → package root
REPO_ROOT = Path("../..").resolve()

# ---------------------------------------------------------------------------
# Data paths
# - DEMO (default): examples/minimal/{Heavy,Light,Paired}/Demo
# - FULL: data/{Heavy,Light,Paired}
# ---------------------------------------------------------------------------
USE_FULL_DB = False

DEMO_ROOT = REPO_ROOT / "examples" / "minimal"
DEMO_HEAVY = DEMO_ROOT / "Heavy" / "Demo"
DEMO_LIGHT = DEMO_ROOT / "Light" / "Demo"
DEMO_PAIRED = DEMO_ROOT / "Paired" / "Demo"

FULL_HEAVY = REPO_ROOT / "data" / "Heavy"
FULL_LIGHT = REPO_ROOT / "data" / "Light"
FULL_PAIRED = REPO_ROOT / "data" / "Paired"

if USE_FULL_DB:
    heavy_dir = FULL_HEAVY
    light_dir = FULL_LIGHT
    paired_dir = FULL_PAIRED
else:
    heavy_dir = DEMO_HEAVY
    light_dir = DEMO_LIGHT
    paired_dir = DEMO_PAIRED


def sequence_parquet_files(data_dir: Path) -> list[str]:
    """All *.parquet under data_dir except metadata.parquet."""
    return sorted(
        str(p) for p in Path(data_dir).rglob("*.parquet") if p.name != "metadata.parquet"
    )


def sql_file_list(files: list[str]) -> str:
    """Format paths for read_parquet([...])."""
    return ", ".join(f"'{f}'" for f in files)


heavy_files = sequence_parquet_files(heavy_dir)
light_files = sequence_parquet_files(light_dir) if light_dir is not None else []
paired_files = sequence_parquet_files(paired_dir) if paired_dir is not None else []

print("USE_FULL_DB:", USE_FULL_DB)
print("heavy_dir:", heavy_dir.relative_to(REPO_ROOT), "exists=", heavy_dir.exists(), "n_files=", len(heavy_files))
if light_dir is not None:
    print("light_dir:", light_dir.relative_to(REPO_ROOT), "exists=", light_dir.exists(), "n_files=", len(light_files))
if paired_dir is not None:
    print("paired_dir:", paired_dir.relative_to(REPO_ROOT), "exists=", paired_dir.exists(), "n_files=", len(paired_files))

con = duckdb.connect()
print("DuckDB ready.")

USE_FULL_DB: False
heavy_dir: examples/minimal/Heavy/Demo exists= True n_files= 1
light_dir: examples/minimal/Light/Demo exists= True n_files= 2
paired_dir: examples/minimal/Paired/Demo exists= True n_files= 1
DuckDB ready.


## 1. Gene filters (V / D / J)

In raw SQL use `LIKE` on the stored IMGT-style names (`IGHV1-3*04`, …).
ASE accepts short forms (`"1-3"`); SQL needs the full prefix.

The minimal Heavy demo is a single-gene subsample (`IGHV1-3*`).

In [15]:
# Equivalent to: engine.search(chain_mode="heavy", heavy_v="1-3")
heavy_list = sql_file_list(heavy_files)

df = con.execute(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet([{heavy_list}])
    WHERE v_call LIKE 'IGHV1-3%'
""").df()
display(df)

,n
0,50


In [16]:
# Family-level V + J  (ASE: heavy_v="1", heavy_j="4")
df = con.execute(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet([{heavy_list}])
    WHERE v_call LIKE 'IGHV1-%'
      AND j_call LIKE 'IGHJ4%'
""").df()
display(df)

,n
0,18


In [17]:
# Multi-parameter: V + J + CDRH3 length
# ASE: heavy_v="1-3", heavy_j="4", heavy_cdr3_length="14"
df = con.execute(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet([{heavy_list}])
    WHERE v_call LIKE 'IGHV1-3%'
      AND j_call LIKE 'IGHJ4%'
      AND cdr3_length = 14
""").df()
display(df)

,n
0,6


## 2. CDR length filters

Lengths are precomputed columns (`cdr3_length`, or `cdr3_length_heavy` / `cdr3_length_light` in paired).

In [18]:
# ASE: heavy_cdr3_length="15-20"
df = con.execute(f"""
    SELECT cdr3_length, COUNT(*) AS n
    FROM read_parquet([{heavy_list}])
    WHERE cdr3_length BETWEEN 15 AND 20
    GROUP BY cdr3_length
    ORDER BY cdr3_length
""").df()
display(df)

,cdr3_length,n
0,15,1
1,16,2
2,17,3
3,18,5
4,19,4
5,20,3


## 3. CDR motif filters (DuckDB regex)

Prefer `regexp_matches(col, pattern)`. ASE motif `AR*` roughly maps to `^AR`.
ASE similarity / mismatch budgets are **not** built into raw SQL — build the regex yourself or use ASE.

In [19]:
# ASE: heavy_v="1-3", heavy_cdr3_motif="AR*"
df = con.execute(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet([{heavy_list}])
    WHERE v_call LIKE 'IGHV1-3%'
      AND regexp_matches(cdr3_aa, '^AR')
""").df()
display(df)

,n
0,39


In [20]:
# Multi-parameter:
# ASE: heavy_v="1-3", heavy_j="4", heavy_cdr3_length="14", heavy_cdr3_motif="AR*"
df = con.execute(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet([{heavy_list}])
    WHERE v_call LIKE 'IGHV1-3%'
      AND j_call LIKE 'IGHJ4%'
      AND cdr3_length = 14
      AND regexp_matches(cdr3_aa, '^AR')
""").df()
display(df)

,n
0,1


## 4. Retrieve sequence rows

ASE: `full_results=True, limit=100`. In SQL: `SELECT … LIMIT 100`.

In [21]:
seqs = con.execute(f"""
    SELECT v_call, j_call, cdr3_aa, cdr3_length
    FROM read_parquet([{heavy_list}])
    WHERE v_call LIKE 'IGHV1-3%'
    LIMIT 100
""").df()
print("rows:", len(seqs))
display(seqs.head())

rows: 50


,v_call,j_call,cdr3_aa,cdr3_length
0,IGHV1-3*04,IGHJ6*02,ARDTSSVKRLRPSSHTYYDFWSGYYSNYYYYGMDV,35
1,IGHV1-3*04,IGHJ6*02,ARDTSSVKRLRPSSHTYYDFWSGYYSNYYYYGMDV,35
2,IGHV1-3*04,IGHJ6*02,ERDTSSVKRLRPSSHTYYDFWSGYYSNYYYYGMDV,35
3,IGHV1-3*04,IGHJ6*02,ARDTSSVKRLRPSSHTYYDFWSGYYSNYYYYGMDV,35
4,IGHV1-3*04,IGHJ6*02,ARDTSSVKRLRPSSHTYYDFWSGYYSNYYYYGMDV,35


## 5. Per-donor hit counts (metadata join)

ASE always returns `stats_df` with donor depth. Sequence Parquet rows typically lack `subject`; join to `metadata.parquet` using DuckDB’s `filename=true` option against `metadata.file_path`.

In [22]:
meta_path = heavy_dir / "metadata.parquet"
stats_df = con.execute(f"""
    SELECT
        m.subject,
        COUNT(*) AS hits,
        ANY_VALUE(m.total_sequences) AS total_sequences,
        COUNT(*) * 1e6 / NULLIF(ANY_VALUE(m.total_sequences), 0) AS per_million
    FROM read_parquet([{heavy_list}], filename = true) s
    JOIN read_parquet('{meta_path}') m
      ON ends_with(replace(s.filename, '\\', '/'), m.file_path)
    WHERE s.v_call LIKE 'IGHV1-3%'
    GROUP BY m.subject
    ORDER BY hits DESC
""").df()

display(stats_df.head(10))

,subject,hits,total_sequences,per_million
0,no,50,50,1000000.0


## 6. Light-chain and paired searches

Works with the minimal Demo folders, or set `USE_FULL_DB = True` for the full Light / Paired trees.

In [23]:
if light_dir is None or not Path(light_dir).exists() or not light_files:
    print("Skip light example: set USE_FULL_DB=True (or ensure DEMO light_dir exists).")
else:
    light_list = sql_file_list(light_files)
    # ASE: chain_mode="light", light_v="L2"  → lambda family 2 only
    df = con.execute(f"""
        SELECT COUNT(*) AS n
        FROM read_parquet([{light_list}])
        WHERE v_call LIKE 'IGLV2-%'
    """).df()
    print("IGLV2* hits:")
    display(df)

IGLV2* hits:


,n
0,14


In [24]:
if paired_dir is None or not Path(paired_dir).exists() or not paired_files:
    print("Skip paired example: set USE_FULL_DB=True (or ensure DEMO paired_dir exists).")
else:
    paired_list = sql_file_list(paired_files)
    # ASE-like paired filter (patterns chosen to hit the tiny Paired demo)
    df = con.execute(f"""
        SELECT COUNT(*) AS n
        FROM read_parquet([{paired_list}])
        WHERE v_call_heavy LIKE 'IGHV4-%'
          AND cdr3_length_heavy BETWEEN 15 AND 20
    """).df()
    print("paired IGHV4* × CDRH3 15–20 hits:")
    display(df)

paired IGHV4* × CDRH3 15–20 hits:


,n
0,6


## 7. Paired IGVH×IG(K/L)V co-occurrence (custom aggregation)

This is beyond the ASE search mask — typical reason to use raw SQL.

In [25]:
if paired_dir is None or not Path(paired_dir).exists() or not paired_files:
    print("Skip VH×VL co-occurrence: set USE_FULL_DB=True (or ensure DEMO paired_dir exists).")
else:
    paired_list = sql_file_list(paired_files)
    vh_vl = con.execute(f"""
        SELECT
            v_call_heavy AS vh,
            v_call_light AS vl,
            COUNT(*) AS count
        FROM read_parquet([{paired_list}])
        WHERE v_call_heavy IS NOT NULL
          AND v_call_light IS NOT NULL
        GROUP BY v_call_heavy, v_call_light
        ORDER BY count DESC
    """).df()
    print("unique allele pairs:", len(vh_vl))
    print("total paired sequences:", int(vh_vl["count"].sum()))
    display(vh_vl.head(15))

unique allele pairs: 37
total paired sequences: 40


,vh,vl,count
0,IGHV4-31*03,IGKV1-39*01,3
1,IGHV4-30-2*01,IGKV1-39*01,2
2,IGHV3-15*01,IGKV2-28*01,1
3,IGHV4-59*02,IGLV3-21*04,1
4,IGHV2-70*01,IGKV1-9*01,1
5,IGHV3-30*18,IGKV4-1*01,1
6,IGHV3-7*05,IGKV3-20*01,1
7,IGHV1-2*04,IGLV1-47*01,1
8,IGHV4-39*07,IGLV1-40*01,1
9,IGHV1-3*01,IGKV2-24*01,1


In [26]:
con.close()
print("Done.")

Done.
